In [31]:
import pymysql


db = pymysql.connect(
    host="localhost",
    port=3306,
    user="root",
    passwd="",
    db="w_db_V1",
    charset="utf8mb4",       
    autocommit=False            
)

try:

    cursor = db.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INT AUTO_INCREMENT PRIMARY KEY,
            product_name  VARCHAR(100),
            brand_name    VARCHAR(100) NOT NULL,
            price         VARCHAR(100),
            review_rate   VARCHAR(100),
            review_count  VARCHAR(100)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)


    db.commit()

finally:
    cursor.close()
    db.close()

In [38]:
cursor.close()

In [24]:
db.commit()

In [47]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
import pickle
import time as t

options = uc.ChromeOptions()
#options.add_argument('--headless')
#options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument('--window-size=1920x1080')
options.add_argument('--start-maximized')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36')
options.add_argument('--lang=ko_KR')

service = Service(ChromeDriverManager().install())
driver = uc.Chrome(service=service, options=options)

db = pymysql.connect(
    host="localhost",
    port=,
    user="",
    passwd="",
    db="w_db_V1",
    charset="utf8mb4",          
    autocommit=False           
)
cursor = db.cursor()

driver.get("https://display.wconcept.co.kr/category/women/002002")
t.sleep(3)

results = []
products = driver.find_elements(By.CSS_SELECTOR,"div.sc-jpuob9-8.iMBqZp.product-item")

for product in products :
    try :
        product_name = product.find_element(By.XPATH, ".//span[contains(@class,'detail')]").text.strip()
        brand_name = product.find_element(By.XPATH, ".//span[contains(@class,'title')]").text.strip()
        price = product.find_element(By.XPATH, ".//span[contains(@class,'final-price')]").text.strip() 
        review_rate = product.find_element(By.XPATH, ".//em[contains(@class,'score')]").text.strip()
        review_count = product.find_element(By.XPATH, ".//span[contains(@class,'cnt')]").text.strip()

        results.append((product_name, brand_name, price, review_rate, review_count))
    except :
        continue

sql = f"""
INSERT INTO products (product_name, brand_name, price, review_rate, review_count)
VALUES (%s, %s, %s, %s, %s)
"""
if results:
    cursor.executemany(sql, results)
    db.commit()
      
driver.quit()
db.commit()
cursor.close()
db.close()
# db.commit()